In [1]:
# ---------- 绘制并导出精美脉络化全局 NACC-CustomKG 图谱 (中文支持) ----------
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from networkx.algorithms import community  # 用于社区检测
import os
import math

# 1. 彻底解决中文乱码与报错问题
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号

# 配置和文件路径
PREFIX = "distmult_NACC"
TSV_FILES = [f"train_{PREFIX}.tsv", f"valid_{PREFIX}.tsv", f"test_{PREFIX}.tsv"]

# 2. 统计变量初始化 (针对 NACC 特征体系分类)
patients = set()
demo_nodes = set()  # 人口学节点 (Age, Gender, Race, etc.)
med_nodes = set()   # EHR/病史节点 (以 his_ 开头的特征)
test_nodes = set()  # BIO/临床测试节点 (APOE, 认知量表, NPIQ, FAQ)
edges = []

count_demo = 0
count_med = 0
count_test = 0

# 人口学关系的精确集合 (对应 NACC EHR 基础部分)
demo_rels = {'has_age_bin', 'has_gender', 'has_education', 'has_hispanic', 'has_race'}

# 3. 读取数据并进行精确分类统计
print(f">> 正在读取 {PREFIX} 图谱文件并分类节点与关系...")
for file in TSV_FILES:
    if os.path.exists(file):
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 3:
                    h, r, t = parts
                    patients.add(h)
                    edges.append((h, t))
                    
                    # 关系与尾实体(特征)同步分类
                    if r in demo_rels:
                        count_demo += 1
                        demo_nodes.add(t)
                    elif r.startswith('has_his_'): # NACC 的病史特征均以 his_ 开头
                        count_med += 1
                        med_nodes.add(t)
                    else:
                        # 剩余的属于 BIO 模块（APOE基因、认知测试量表等）
                        count_test += 1
                        test_nodes.add(t)

# 合并所有特征节点以计算总数
all_features = demo_nodes | med_nodes | test_nodes
total_triples = len(edges)
total_nodes = len(patients) + len(all_features)

print(f"✅ NACC 数据读取完毕！总节点: {total_nodes}, 总三元组/边: {total_triples}")
print(">> 正在构建 NetworkX 图结构...")

# 4. 构建 NetworkX 图
G = nx.Graph()
G.add_nodes_from(patients)
G.add_nodes_from(all_features)
G.add_edges_from(edges)

# 5. 绘图配置与社区布局方案
plt.figure(figsize=(24, 24), dpi=300) 
ax = plt.gca()
ax.set_facecolor('#ffffff') # 纯白背景

# ---------------------------------------------------------
# 基于社区的分层布局 (针对 NACC 这种高维度特征图谱更有效)
# ---------------------------------------------------------
print(">> 正在进行社区检测发现 NACC 患者群落脉络...")
# 使用贪婪模块度社区检测
communities_generator = community.greedy_modularity_communities(G)
communities = sorted(communities_generator, key=len, reverse=True)
num_communities = len(communities)
print(f"✅ 在 NACC 图谱中发现 {num_communities} 个主要社区岛屿！")

# 1. 独立布局社区中心
meta_pos = {}
radius = 14.0 # NACC 数据量大，半径稍微增大以防拥挤
for i in range(num_communities):
    angle = (2 * math.pi * i) / num_communities
    meta_pos[i] = (radius * math.cos(angle), radius * math.sin(angle))

# 2. 局部布局
pos = {}
print(">> 正在为各个临床特征社区进行空间定位...")
for i, comm in enumerate(communities):
    subgraph = G.subgraph(comm)
    sub_pos = nx.spring_layout(subgraph, k=0.12, iterations=15, seed=42, center=meta_pos[i], scale=2.5)
    for node, (x, y) in sub_pos.items():
        pos[node] = (x, y)

print("✅ NACC 全局拓扑脉络布局计算完成。")

# 6. 渲染图形
print(">> 正在渲染精美分类节点与曲线连线...")

# 绘制边 (降低透明度防止线条遮盖节点)
nx.draw_networkx_edges(G, pos, width=0.04, alpha=0.12, edge_color='#999999', connectionstyle='arc3,rad=0.05')

# 渲染分类节点
nx.draw_networkx_nodes(G, pos, nodelist=list(test_nodes), node_color='#2196F3', node_size=16, alpha=0.95, label='BIO 模块 (认知/基因/功能)')
nx.draw_networkx_nodes(G, pos, nodelist=list(med_nodes), node_color='#4CAF50', node_size=16, alpha=0.95, label='EHR 模块 (病史/临床背景)')
nx.draw_networkx_nodes(G, pos, nodelist=list(demo_nodes), node_color='#FFC107', node_size=16, alpha=0.95, label='人口学特征')
nx.draw_networkx_nodes(G, pos, nodelist=list(patients), node_color='#FF4B4B', node_size=10, alpha=0.9, label='NACC 患者实体')

# 7. 左上角统计信息框与图例
stats_text = (
    f"CustomKG Global Topology (NACC)\n"
    f"─ 基于诊断表 NACC 构建的私域知识图谱 ─\n\n"
    f"Total Nodes (节点总数): {total_nodes:,}\n"
    f"  - Patients (患者标识): {len(patients):,}\n"
    f"  - Features (临床特征): {len(all_features):,}\n"
    f"Total Triples (三元组总数): {total_triples:,}\n\n"
    f"Module Breakdown (知识模块分布):\n"
    f" - Demographics (基础背景): {count_demo:,} edges\n"
    f" - Medical History (病史模块): {count_med:,} edges\n"
    f" - BIO / Psychometrics (认知量表): {count_test:,} edges"
)

# 添加文本框
props = dict(boxstyle='round', facecolor='#F8F9FA', alpha=0.95, edgecolor='#CCCCCC')
ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=14,
        verticalalignment='top', bbox=props, fontweight='bold')

# 添加图例标识 (中文)
legend_handles = [
    mpatches.Patch(color='#FF4B4B', label='Patient (患者核心实体)'),
    mpatches.Patch(color='#FFC107', label='Demographics (人口学背景)'),
    mpatches.Patch(color='#4CAF50', label='EHR History (临床病史)'),
    mpatches.Patch(color='#2196F3', label='BIO / Psychometrics (量表/基因测试)')
]
plt.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0.02, 0.78), 
            fontsize=14, framealpha=0.95, edgecolor='#CCCCCC')

plt.axis('off')
plt.tight_layout()

# 8. 导出 PDF
output_pdf = "NACC-CustomKG-Topology.pdf"
print(f">> 正在导出矢量 PDF 至 {output_pdf} ...")
plt.savefig(output_pdf, format='pdf', bbox_inches='tight')
plt.close()
print(f"✅ 导出完成！NACC 私域知识图谱拓扑已生成。")

>> 正在读取 distmult_NACC 图谱文件并分类节点与关系...
✅ NACC 数据读取完毕！总节点: 4003, 总三元组/边: 291673
>> 正在构建 NetworkX 图结构...
>> 正在进行社区检测发现 NACC 患者群落脉络...
✅ 在 NACC 图谱中发现 5 个主要社区岛屿！
>> 正在为各个临床特征社区进行空间定位...
✅ NACC 全局拓扑脉络布局计算完成。
>> 正在渲染精美分类节点与曲线连线...


C:\Users\HP\AppData\Local\Temp\ipykernel_9364\2152852786.py:106: UserWarning: 

The connectionstyle keyword argument is not applicable when drawing edges
with LineCollection.

To make this warning go away, either specify `arrows=True` to
force FancyArrowPatches or use the default values.
Note that using FancyArrowPatches may be slow for large graphs.

  nx.draw_networkx_edges(G, pos, width=0.04, alpha=0.12, edge_color='#999999', connectionstyle='arc3,rad=0.05')


>> 正在导出矢量 PDF 至 NACC-CustomKG-Topology.pdf ...
✅ 导出完成！NACC 私域知识图谱拓扑已生成。


In [1]:
import networkx as nx
from pyvis.network import Network
import os
import random
import warnings

warnings.filterwarnings('ignore')

# 配置和文件路径
PREFIX = "distmult_NACC"
TSV_FILES = [f"train_{PREFIX}.tsv", f"valid_{PREFIX}.tsv", f"test_{PREFIX}.tsv"]

# ⚠️ 注意：由于使用了 Pyvis 交互式图谱，输出格式需要更改为 HTML，在此向您确认该文件名是否可行。
OUTPUT_HTML_PATH = "NACC-CustomKG-Topology.html"

def plot_nacc_custom_kg():
    print("==================================================")
    print("🌐 开始构建多类型均衡的 NACC-CustomKG 连通图谱 (白色背景, 高对比度配色)...")

    # 1. 初始化统计映射与基础集合
    node_type_map = {}
    edges_data = []
    count_demo = 0
    count_med = 0
    count_test = 0

    demo_rels = {'has_age_bin', 'has_gender', 'has_education', 'has_hispanic', 'has_race'}

    # 2. 读取数据并进行精确分类统计
    print(f">> 正在读取 {PREFIX} 图谱文件并分类节点与关系...")
    for file in TSV_FILES:
        if os.path.exists(file):
            with open(file, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) == 3:
                        h, r, t = parts
                        # 头实体固定为患者
                        node_type_map[h] = 'patient'
                        edges_data.append((h, t, r))
                        
                        # 关系与尾实体(特征)同步分类
                        if r in demo_rels:
                            node_type_map[t] = 'demographics'
                            count_demo += 1
                        elif r.startswith('has_his_'):
                            node_type_map[t] = 'medical_history'
                            count_med += 1
                        else:
                            node_type_map[t] = 'bio_psychometrics'
                            count_test += 1

    if not edges_data:
        print("❌ 未读取到任何三元组数据，请检查 TSV 文件。")
        return

    total_triples = len(edges_data)
    total_nodes = len(node_type_map)
    print(f"✅ NACC 数据读取完毕！总节点: {total_nodes}, 总三元组/边: {total_triples}")

    # 3. 构建全量无向图用于寻径
    print(">> 正在构建 NetworkX 图结构...")
    G_full = nx.Graph()
    for h, t, r in edges_data:
        G_full.add_edge(h, t, relation=r)

    # 4. 提取多类型均衡连通子图 (Shortest Path Routing)
    print("⏳ 正在进行多类型路由寻径，确保包含所有类型节点且不断开...")
    largest_cc = max(nx.connected_components(G_full), key=len)
    G_sub = G_full.subgraph(largest_cc)

    central_node = sorted(G_sub.degree, key=lambda x: x[1], reverse=True)[0][0]

    nodes_by_type = {}
    for node in G_sub.nodes():
        ntype = node_type_map.get(node, 'unknown')
        if ntype not in nodes_by_type:
            nodes_by_type[ntype] = []
        nodes_by_type[ntype].append(node)

    random.seed(42) 
    TARGET_PER_TYPE = 60
    core_nodes = set([central_node])

    for ntype, nodes_list in nodes_by_type.items():
        sampled_targets = random.sample(nodes_list, min(TARGET_PER_TYPE, len(nodes_list)))
        for target in sampled_targets:
            try:
                path = nx.shortest_path(G_sub, source=central_node, target=target)
                core_nodes.update(path)
            except nx.NetworkXNoPath:
                continue

    G_core = G_sub.subgraph(core_nodes)

    # 5. 高对比度配色方案与名称映射
    print(">> 正在渲染精美分类节点与曲线连线...")
    color_map = {
        'patient': '#FF4B4B',           # 患者实体
        'demographics': '#FFC107',      # 人口学背景
        'medical_history': '#4CAF50',   # 临床病史
        'bio_psychometrics': '#2196F3'  # 量表/基因测试
    }

    type_name_map = {
        'patient': 'Patient (患者核心实体)',
        'demographics': 'Demographics (人口学背景)',
        'medical_history': 'EHR History (临床病史)',
        'bio_psychometrics': 'BIO / Psychometrics (量表/基因测试)'
    }

    net = Network(height="900px", width="100%", bgcolor="#ffffff", font_color="#333333", select_menu=True)

    # 注入节点
    for node in G_core.nodes():
        ntype = node_type_map.get(node, 'unknown')
        color = color_map.get(ntype, '#A9A9A9')
        display_type = type_name_map.get(ntype, ntype)
        
        # 区分患者与特征节点的尺寸
        node_size = 15 if ntype != 'patient' else 10

        net.add_node(node, 
                     label=" ", 
                     title=f"实体名称: {node}\n实体类型: {display_type}",
                     color=color,
                     size=node_size)

    # 注入连线
    for u, v, data in G_core.edges(data=True):
        net.add_edge(u, v, 
                     title=data.get('relation', ''), 
                     label=' ', 
                     width=0.5,         
                     color='#bdc3c7')   

    net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=150, spring_strength=0.05, damping=0.4, overlap=0)
    net.write_html(OUTPUT_HTML_PATH)

    # 6. 注入全局统计面板
    patient_count = sum(1 for v in node_type_map.values() if v == 'patient')
    feature_count = total_nodes - patient_count

    legend_html = f"""
    <div style="position: absolute; top: 20px; left: 20px; z-index: 999; background-color: rgba(255, 255, 255, 0.95); color: #333; padding: 15px 25px; border-radius: 8px; border: 1px solid #ddd; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
        <h3 style="margin-top: 0; margin-bottom: 10px; font-size: 16px; border-bottom: 2px solid #eee; padding-bottom: 5px; color: #2c3e50;">CustomKG Global Topology (NACC)</h3>
        <p style="font-size: 12px; color: #666; margin-top: 0; margin-bottom: 15px;">─ 基于诊断表 NACC 构建的私域知识图谱 ─</p>
        
        <div style="font-size: 13px; margin-bottom: 15px; border-bottom: 1px solid #eee; padding-bottom: 10px;">
            <div><b>Total Nodes:</b> {total_nodes:,}</div>
            <div style="margin-left: 10px; color: #555;">- Patients: {patient_count:,}</div>
            <div style="margin-left: 10px; color: #555;">- Features: {feature_count:,}</div>
            <div style="margin-top: 5px;"><b>Total Triples:</b> {total_triples:,}</div>
        </div>
        
        <h4 style="margin: 0 0 10px 0; font-size: 14px;">Module Breakdown:</h4>
    """
    
    # 按照原设定分类展示边数与图例
    stats_data = [
        ('demographics', count_demo),
        ('medical_history', count_med),
        ('bio_psychometrics', count_test)
    ]

    for ntype, count in stats_data:
        color = color_map.get(ntype, '#A9A9A9')
        display_name = type_name_map.get(ntype, ntype)
        legend_html += f'<div style="margin: 8px 0; font-size: 13px; display: flex; align-items: center;"><span style="display: inline-block; width: 14px; height: 14px; background-color: {color}; border-radius: 50%; margin-right: 12px; border: 1px solid #888;"></span><span style="font-weight: 500; width: 220px;">{display_name}</span> <span>{count:,} edges</span></div>'
    
    # 单独展示 Patient 颜色图示
    patient_color = color_map['patient']
    legend_html += f'<div style="margin: 8px 0; font-size: 13px; display: flex; align-items: center;"><span style="display: inline-block; width: 14px; height: 14px; background-color: {patient_color}; border-radius: 50%; margin-right: 12px; border: 1px solid #888;"></span><span style="font-weight: 500; width: 220px;">{type_name_map["patient"]}</span></div>'

    legend_html += "</div>"

    with open(OUTPUT_HTML_PATH, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    html_content = html_content.replace('<body>', f'<body>\n{legend_html}')
    
    with open(OUTPUT_HTML_PATH, 'w', encoding='utf-8') as f:
        f.write(html_content)

    print(f"💾 NACC 私域知识图谱拓扑已生成，文件保存至: {OUTPUT_HTML_PATH}")
    print("==================================================")

if __name__ == "__main__":
    plot_nacc_custom_kg()

🌐 开始构建多类型均衡的 NACC-CustomKG 连通图谱 (白色背景, 高对比度配色)...
>> 正在读取 distmult_NACC 图谱文件并分类节点与关系...
✅ NACC 数据读取完毕！总节点: 4003, 总三元组/边: 291673
>> 正在构建 NetworkX 图结构...
⏳ 正在进行多类型路由寻径，确保包含所有类型节点且不断开...
>> 正在渲染精美分类节点与曲线连线...
💾 NACC 私域知识图谱拓扑已生成，文件保存至: NACC-CustomKG-Topology.html
